# Optimización Final con Optuna

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Optimizar hiperparámetros de los 3 mejores modelos del notebook 02:
1. **Logistic Regression + Top-5 features** (ROC-AUC baseline: 0.511)
2. **XGBoost + PCA-15** (ROC-AUC baseline: 0.507)
3. **Random Forest + PCA-5** (ROC-AUC baseline: 0.504)

**Métrica de optimización:** ROC-AUC (más robusta que F1-Score)

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import joblib
import warnings
warnings.filterwarnings('ignore')

d:\Documents\jcbl\MIAD 2025 ciclo 4\Curso Despliegue Soluciones Analiticas\Proyecto\Despliegue-Soluciones-Proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración

In [2]:
# Configurar MLflow
experiment_name = "/sp500-optuna-optimization"
mlflow.set_experiment(experiment_name)

print(f"Experimento: {experiment_name}")

Experimento: /sp500-optuna-optimization


## 2. Carga de Datos

In [ ]:
# Cargar datasets
train = pd.read_parquet('./data/processed/ml_ready/train.parquet')
test = pd.read_parquet('./data/processed/ml_ready/test.parquet')

# Separar features y target
feature_cols = [col for col in train.columns if col not in ['Ticker', 'Date', 'Target']]

X_train_full = train[feature_cols]
y_train = train['Target']
X_test_full = test[feature_cols]
y_test = test['Target']

print(f"Datos cargados: {X_train_full.shape[0]} train, {X_test_full.shape[0]} test")
print(f"Features: {len(feature_cols)}")

Datos cargados: 20128 train, 5032 test
Features: 22


## 3. Preparar las 3 Configuraciones

Basadas en los mejores resultados del notebook 02.

In [4]:
# Top-5 features (para Logistic Regression)
from sklearn.ensemble import RandomForestClassifier as RFC
rf_temp = RFC(random_state=42)
rf_temp.fit(X_train_full, y_train)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

top_5_features = feature_importance.head(5)['feature'].tolist()

print("Top-5 features:")
print(top_5_features)

Top-5 features:
['Volume_change', 'Returns', 'RSI_14', 'BB_width', 'MACD_diff']


In [5]:
# Configuración 1: Logistic Regression + Top-5
X_train_lr = X_train_full[top_5_features]
X_test_lr = X_test_full[top_5_features]

print(f"Config 1 - Logistic Regression: {X_train_lr.shape}")

# Configuración 2: XGBoost + PCA-15
scaler_xgb = StandardScaler()
X_train_scaled_xgb = scaler_xgb.fit_transform(X_train_full)
X_test_scaled_xgb = scaler_xgb.transform(X_test_full)

pca_xgb = PCA(n_components=15, random_state=42)
X_train_xgb = pca_xgb.fit_transform(X_train_scaled_xgb)
X_test_xgb = pca_xgb.transform(X_test_scaled_xgb)

print(f"Config 2 - XGBoost: {X_train_xgb.shape}, Varianza: {pca_xgb.explained_variance_ratio_.sum():.2%}")

# Configuración 3: Random Forest + PCA-5
scaler_rf = StandardScaler()
X_train_scaled_rf = scaler_rf.fit_transform(X_train_full)
X_test_scaled_rf = scaler_rf.transform(X_test_full)

pca_rf = PCA(n_components=5, random_state=42)
X_train_rf = pca_rf.fit_transform(X_train_scaled_rf)
X_test_rf = pca_rf.transform(X_test_scaled_rf)

print(f"Config 3 - Random Forest: {X_train_rf.shape}, Varianza: {pca_rf.explained_variance_ratio_.sum():.2%}")

Config 1 - Logistic Regression: (20128, 5)
Config 2 - XGBoost: (20128, 15), Varianza: 100.00%
Config 3 - Random Forest: (20128, 5), Varianza: 86.07%


## 4. Optimización 1: Logistic Regression + Top-5

In [6]:
def objective_lr(trial):
    """Optuna objective para Logistic Regression."""
    params = {
        'C': trial.suggest_float('C', 0.001, 100, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver': 'saga',
        'max_iter': 1000,
        'random_state': 42
    }
    
    model = LogisticRegression(**params)
    
    # Cross-validation con ROC-AUC
    cv_scores = cross_val_score(
        model, X_train_lr, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando Logistic Regression + Top-5...")
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=50, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_lr.best_value:.4f}")
print(f"Mejores parámetros: {study_lr.best_params}")

[I 2025-11-22 22:33:54,573] A new study created in memory with name: no-name-c0a80d58-cd55-44d4-a3c6-a554b04031d0


Optimizando Logistic Regression + Top-5...


Best trial: 0. Best value: 0.506079:   2%|▏         | 1/50 [00:13<10:42, 13.11s/it]

[I 2025-11-22 22:34:07,718] Trial 0 finished with value: 0.5060790373464051 and parameters: {'C': 6.722388790064812, 'penalty': 'l2'}. Best is trial 0 with value: 0.5060790373464051.


Best trial: 1. Best value: 0.506116:   4%|▍         | 2/50 [00:20<07:49,  9.78s/it]

...

[I 2025-11-22 22:35:50,675] Trial 47 finished with value: 0.5041371098448355 and parameters: {'C': 0.03746436457394312, 'penalty': 'l1'}. Best is trial 7 with value: 0.5094241218118204.


Best trial: 7. Best value: 0.509424:  98%|█████████▊| 49/50 [02:05<00:03,  3.96s/it]

[I 2025-11-22 22:36:00,180] Trial 48 finished with value: 0.5061138536849827 and parameters: {'C': 30.68391729470391, 'penalty': 'l2'}. Best is trial 7 with value: 0.5094241218118204.


Best trial: 7. Best value: 0.509424: 100%|██████████| 50/50 [02:07<00:00,  2.55s/it]

[I 2025-11-22 22:36:02,097] Trial 49 finished with value: 0.5094241069447263 and parameters: {'C': 0.00283851059144879, 'penalty': 'l1'}. Best is trial 7 with value: 0.5094241218118204.

Mejor ROC-AUC (CV): 0.5094
Mejores parámetros: {'C': 0.007960411345734336, 'penalty': 'l1'}


## 5. Optimización 2: XGBoost + PCA-15

In [7]:
def objective_xgb(trial):
    """Optuna objective para XGBoost."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    
    cv_scores = cross_val_score(
        model, X_train_xgb, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando XGBoost + PCA-15...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_xgb.best_value:.4f}")
print(f"Mejores parámetros: {study_xgb.best_params}")

[I 2025-11-22 22:36:02,135] A new study created in memory with name: no-name-47211b42-1a96-4739-adf5-b7cfc72a2a8e


Optimizando XGBoost + PCA-15...


  0%|          | 0/100 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.511282:   1%|          | 1/100 [00:01<02:09,  1.31s/it]

[I 2025-11-22 22:36:03,439] Trial 0 finished with value: 0.5112816832099649 and parameters: {'n_estimators': 128, 'max_depth': 6, 'learning_rate': 0.11617349874495107, 'subsample': 0.8730142123886586, 'colsample_bytree': 0.9954117872459898, 'min_child_weight': 7, 'gamma': 4.438195048964212}. Best is trial 0 with value: 0.5112816832099649.


...

[I 2025-11-22 22:41:33,145] Trial 97 finished with value: 0.5094296338385194 and parameters: {'n_estimators': 218, 'max_depth': 9, 'learning_rate': 0.07866433647933689, 'subsample': 0.7127631167206357, 'colsample_bytree': 0.865029075158905, 'min_child_weight': 7, 'gamma': 0.314902463971244}. Best is trial 94 with value: 0.516139252573221.


Best trial: 94. Best value: 0.516139:  99%|█████████▉| 99/100 [05:34<00:03,  3.76s/it]

[I 2025-11-22 22:41:36,144] Trial 98 finished with value: 0.5140354122492251 and parameters: {'n_estimators': 207, 'max_depth': 8, 'learning_rate': 0.0635680768262905, 'subsample': 0.8909257474017159, 'colsample_bytree': 0.8873366127063211, 'min_child_weight': 8, 'gamma': 0.41658235815208844}. Best is trial 94 with value: 0.516139252573221.


Best trial: 94. Best value: 0.516139: 100%|██████████| 100/100 [05:38<00:00,  3.38s/it]

[I 2025-11-22 22:41:40,360] Trial 99 finished with value: 0.513213395746757 and parameters: {'n_estimators': 229, 'max_depth': 9, 'learning_rate': 0.04991670269880252, 'subsample': 0.9074119273937551, 'colsample_bytree': 0.9040561410214605, 'min_child_weight': 7, 'gamma': 0.19163309146937324}. Best is trial 94 with value: 0.516139252573221.

Mejor ROC-AUC (CV): 0.5161
Mejores parámetros: {'n_estimators': 214, 'max_depth': 9, 'learning_rate': 0.07264906958124179, 'subsample': 0.9079766089894659, 'colsample_bytree': 0.891998964087873, 'min_child_weight': 7, 'gamma': 0.24250168874868827}


## 6. Optimización 3: Random Forest + PCA-5

In [8]:
def objective_rf(trial):
    """Optuna objective para Random Forest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**params)
    
    cv_scores = cross_val_score(
        model, X_train_rf, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando Random Forest + PCA-5...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_rf.best_value:.4f}")
print(f"Mejores parámetros: {study_rf.best_params}")

[I 2025-11-22 22:41:40,391] A new study created in memory with name: no-name-b5929e8b-4bf3-40a4-83a9-672a9b10a7a4


Optimizando Random Forest + PCA-5...


Best trial: 0. Best value: 0.504177:   2%|▏         | 1/50 [00:10<08:51, 10.85s/it]

[I 2025-11-22 22:41:51,233] Trial 0 finished with value: 0.5041773147218487 and parameters: {'n_estimators': 135, 'max_depth': 20, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.5041773147218487.


Best trial: 1. Best value: 0.505538:   4%|▍         | 2/50 [00:15<05:49,  7.27s/it]

...

[I 2025-11-22 22:47:17,360] Trial 47 finished with value: 0.5163781767486957 and parameters: {'n_estimators': 68, 'max_depth': 5, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 34 with value: 0.5168473192480835.


Best trial: 34. Best value: 0.516847:  98%|█████████▊| 49/50 [05:39<00:02,  2.57s/it]

[I 2025-11-22 22:47:19,981] Trial 48 finished with value: 0.5121691834054745 and parameters: {'n_estimators': 68, 'max_depth': 9, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 34 with value: 0.5168473192480835.


Best trial: 34. Best value: 0.516847: 100%|██████████| 50/50 [05:45<00:00,  6.91s/it]

[I 2025-11-22 22:47:25,828] Trial 49 finished with value: 0.5074116498960913 and parameters: {'n_estimators': 91, 'max_depth': 19, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 34 with value: 0.5168473192480835.

Mejor ROC-AUC (CV): 0.5168
Mejores parámetros: {'n_estimators': 73, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


## 7. Comparación de los 3 Modelos Optimizados

Entrenar cada modelo con sus mejores hiperparámetros y evaluar en test set.

In [9]:
# Entrenar y evaluar los 3 modelos optimizados
resultados_optuna = []

# Modelo 1: Logistic Regression
params_lr = study_lr.best_params.copy()
params_lr.update({'solver': 'saga', 'max_iter': 1000, 'random_state': 42})
model_lr = LogisticRegression(**params_lr)
model_lr.fit(X_train_lr, y_train)

y_pred_lr = model_lr.predict(X_test_lr)
y_pred_proba_lr = model_lr.predict_proba(X_test_lr)[:, 1]

resultados_optuna.append({
    'Modelo': 'Logistic Regression',
    'Config': 'Top-5',
    'ROC-AUC (CV)': study_lr.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_lr),
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'F1-Score': f1_score(y_test, y_pred_lr)
})

# Modelo 2: XGBoost
params_xgb = study_xgb.best_params.copy()
params_xgb.update({'random_state': 42, 'eval_metric': 'logloss'})
model_xgb = XGBClassifier(**params_xgb)
model_xgb.fit(X_train_xgb, y_train)

y_pred_xgb = model_xgb.predict(X_test_xgb)
y_pred_proba_xgb = model_xgb.predict_proba(X_test_xgb)[:, 1]

resultados_optuna.append({
    'Modelo': 'XGBoost',
    'Config': 'PCA-15',
    'ROC-AUC (CV)': study_xgb.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_xgb),
    'Accuracy': accuracy_score(y_test, y_pred_xgb),
    'F1-Score': f1_score(y_test, y_pred_xgb)
})

# Modelo 3: Random Forest
params_rf = study_rf.best_params.copy()
params_rf.update({'random_state': 42, 'n_jobs': -1})
model_rf = RandomForestClassifier(**params_rf)
model_rf.fit(X_train_rf, y_train)

y_pred_rf = model_rf.predict(X_test_rf)
y_pred_proba_rf = model_rf.predict_proba(X_test_rf)[:, 1]

resultados_optuna.append({
    'Modelo': 'Random Forest',
    'Config': 'PCA-5',
    'ROC-AUC (CV)': study_rf.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_rf),
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'F1-Score': f1_score(y_test, y_pred_rf)
})

# Mostrar comparación
df_resultados = pd.DataFrame(resultados_optuna)
df_resultados = df_resultados.sort_values('ROC-AUC (Test)', ascending=False).reset_index(drop=True)

print(df_resultados.to_string(index=False))

             Modelo Config  ROC-AUC (CV)  ROC-AUC (Test)  Accuracy  F1-Score
      Random Forest  PCA-5      0.516847        0.514381  0.523649  0.663011
Logistic Regression  Top-5      0.509424        0.510683  0.516494  0.681169
            XGBoost PCA-15      0.516139        0.494301  0.495231  0.554073


## 8. Resumen Final

Mejor modelo: XGBoost con PCA-15   

In [10]:
# Modelo ganador: XGBoost + PCA-15
print(f"\nMétricas de Optimización (Cross-Validation):")
print(f"  ROC-AUC (CV): {df_resultados.loc[0, 'ROC-AUC (CV)']:.4f}")
print(f"\nMétricas en Test Set:")
print(f"  ROC-AUC:   {df_resultados.loc[0, 'ROC-AUC (Test)']:.4f}")
print(f"  Accuracy:  {df_resultados.loc[0, 'Accuracy']:.4f}")
print(f"  F1-Score:  {df_resultados.loc[0, 'F1-Score']:.4f}")

# Matriz de Confusión del modelo XGBoost
print("Matriz de Confusión:")
cm = confusion_matrix(y_test, y_pred_xgb)
print(cm)
print(f"\nTrue Negatives (TN):  {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP):  {cm[1,1]}")

# Classification Report detallado
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Vender (0)', 'Comprar (1)']))


Métricas de Optimización (Cross-Validation):
  ROC-AUC (CV): 0.5168

Métricas en Test Set:
  ROC-AUC:   0.5144
  Accuracy:  0.5236
  F1-Score:  0.6630
Matriz de Confusión:
[[ 914 1519]
 [1021 1578]]

True Negatives (TN):  914
False Positives (FP): 1519
False Negatives (FN): 1021
True Positives (TP):  1578

Classification Report:
              precision    recall  f1-score   support

  Vender (0)       0.47      0.38      0.42      2433
 Comprar (1)       0.51      0.61      0.55      2599

    accuracy                           0.50      5032
   macro avg       0.49      0.49      0.49      5032
weighted avg       0.49      0.50      0.49      5032

